# TemporalRec — Data Quality and Cleaning

**Current scope:** exact-duplicate removal and timestamp-tie handling only. We do **not** yet make training labels, apply the final user/item 5-core filter, or engineer model features.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

# The notebook must be started with the repository as the current folder.
PROJECT_ROOT = Path.cwd(
RAW_EVENTS = PROJECT_ROOT / "dataset" / "events.csv"

# Change this name before a second full run. It creates a new, separate audit folder.
RUN_TAG = "notebook_run_2026_10_13"
REPORT_DIR = PROJECT_ROOT / "reports" / RUN_TAG
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / RUN_TAG

assert RAW_EVENTS.exists(), f"Cannot find {RAW_EVENTS}. Start JupyterLab from the project root."
assert not REPORT_DIR.exists(), (
    f"{REPORT_DIR} already exists. Change RUN_TAG rather than overwriting an old audit."
)
REPORT_DIR.mkdir(parents=True)
INTERIM_DIR.mkdir(parents=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw input:    {RAW_EVENTS}")
print(f"Audit output: {REPORT_DIR}")
print(f"Clean output: {INTERIM_DIR}")

AssertionError: Cannot find E:\ALL MCP Servers\Recommendation\notebooks\dataset\events.csv. Start JupyterLab from the project root.

In [ ]:
# A checksum helps prove exactly which raw file this run used.
def sha256_file(path: Path, chunk_size: int = 1_048_576) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

run_metadata = {
    "run_tag": RUN_TAG,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "raw_file": str(RAW_EVENTS.relative_to(PROJECT_ROOT)),
    "raw_sha256": sha256_file(RAW_EVENTS),
}
pd.Series(run_metadata, name="value").to_csv(REPORT_DIR / "run_metadata.csv")
run_metadata

## 1. Load the raw events

We explicitly set data types. `transactionid` is nullable because it is naturally empty for views and add-to-cart events. We do **not** fill those blanks.

In [ ]:
EVENT_COLUMNS = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
ID_DTYPES = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "category",
    "itemid": "int64",
    "transactionid": "Int64",
}

raw = pd.read_csv(RAW_EVENTS, dtype=ID_DTYPES)
assert raw.columns.tolist() == EVENT_COLUMNS, "Unexpected column names or order. Stop and inspect the raw file."
raw.info()
raw.head()

## 2. Basic assumptions

A recommendation sequence needs a visitor ID, item ID, event name, and timestamp. This checks those fields before cleaning anything.

In [ ]:
EXPECTED_EVENTS = {"view", "addtocart", "transaction"}
required_columns = ["timestamp", "visitorid", "event", "itemid"]
transaction_mask = raw["event"].eq("transaction")

assumptions = pd.DataFrame([
    {
        "check": "Required fields have no blanks",
        "value": int(raw[required_columns].isna().sum().sum()),
        "expected": 0,
        "passed": int(raw[required_columns].isna().sum().sum()) == 0,
    },
    {
        "check": "Only expected event names occur",
        "value": ", ".join(sorted(set(raw["event"].dropna().astype(str)) - EXPECTED_EVENTS)),
        "expected": "empty string",
        "passed": set(raw["event"].dropna().astype(str)) <= EXPECTED_EVENTS,
    },
    {
        "check": "Every transaction has transactionid",
        "value": int(raw.loc[transaction_mask, "transactionid"].isna().sum()),
        "expected": 0,
        "passed": int(raw.loc[transaction_mask, "transactionid"].isna().sum()) == 0,
    },
    {
        "check": "Non-transaction rows have blank transactionid",
        "value": int(raw.loc[~transaction_mask, "transactionid"].notna().sum()),
        "expected": 0,
        "passed": int(raw.loc[~transaction_mask, "transactionid"].notna().sum()) == 0,
    },
])
assumptions.to_csv(REPORT_DIR / "01_assumption_checks.csv", index=False)
assumptions

## 3. Exact duplicates

An exact duplicate means all five raw fields are the same. We retain the earliest source row and remove only later identical copies in a **new** file.

In [ ]:
working = raw.copy()
working["source_row_number"] = working.index + 2  # Row 1 is the CSV header in Excel.

duplicate_member = working.duplicated(EVENT_COLUMNS, keep=False)
exact_duplicate_audit = working.loc[duplicate_member, ["source_row_number", *EVENT_COLUMNS]].copy()
exact_duplicate_audit = exact_duplicate_audit.sort_values(EVENT_COLUMNS + ["source_row_number"], kind="stable")
duplicate_group = exact_duplicate_audit.groupby(EVENT_COLUMNS, dropna=False, sort=False)
exact_duplicate_audit["duplicate_group_id"] = duplicate_group.ngroup() + 1
exact_duplicate_audit["duplicate_rank_in_group"] = duplicate_group.cumcount() + 1
exact_duplicate_audit["duplicate_group_size"] = duplicate_group["source_row_number"].transform("size")
exact_duplicate_audit["action"] = exact_duplicate_audit["duplicate_rank_in_group"].map(
    lambda rank: "keep_first_source_row" if rank == 1 else "remove_exact_duplicate"
)
exact_duplicate_audit = exact_duplicate_audit[[
    "duplicate_group_id", "duplicate_rank_in_group", "duplicate_group_size",
    "action", "source_row_number", *EVENT_COLUMNS,
]]

duplicate_group_summary = (
    exact_duplicate_audit[["duplicate_group_id", "duplicate_group_size"]]
    .drop_duplicates()
    .groupby("duplicate_group_size", as_index=False)
    .size()
    .rename(columns={"size": "duplicate_groups"})
)
duplicate_group_summary["rows_in_groups"] = (
    duplicate_group_summary["duplicate_group_size"] * duplicate_group_summary["duplicate_groups"]
)

exact_duplicate_audit.to_csv(REPORT_DIR / "02_exact_duplicate_audit.csv", index=False)
duplicate_group_summary.to_csv(REPORT_DIR / "02_exact_duplicate_summary.csv", index=False)
duplicate_group_summary

In [ ]:
# This creates a separate file. `raw` is still unchanged in memory and on disk.
exact_deduped = working.drop(columns="source_row_number").drop_duplicates(EVENT_COLUMNS, keep="first").copy()
exact_deduped_path = INTERIM_DIR / "events_exact_deduplicated.csv"
exact_deduped.to_csv(exact_deduped_path, index=False)

removed_exact_copies = len(raw) - len(exact_deduped)
assert exact_deduped.duplicated(EVENT_COLUMNS, keep=False).sum() == 0
print(f"Raw rows: {len(raw):,}")
print(f"Exact copies removed: {removed_exact_copies:,}")
print(f"Rows in separate cleaned file: {len(exact_deduped):,}")

## 4. Repeated visitor–item interactions

This is **not** duplicate removal. A visitor can truly view the same item several times. We measure these repeats and keep them.

In [ ]:
repeated_pair_summary = (
    exact_deduped.groupby(["visitorid", "itemid"], as_index=False)
    .agg(
        interaction_count=("itemid", "size"),
        first_timestamp_ms=("timestamp", "min"),
        last_timestamp_ms=("timestamp", "max"),
        distinct_event_types=("event", "nunique"),
    )
)
repeated_pair_summary = repeated_pair_summary.query("interaction_count > 1").copy()
repeated_pair_summary["time_span_hours"] = (
    repeated_pair_summary["last_timestamp_ms"] - repeated_pair_summary["first_timestamp_ms"]
) / 3_600_000
repeated_pair_summary = repeated_pair_summary.sort_values(
    ["interaction_count", "visitorid", "itemid"],
    ascending=[False, True, True],
    kind="stable",
)
repeated_pair_summary.to_csv(REPORT_DIR / "03_repeated_visitor_item_summary.csv", index=False)

print(f"Repeated visitor-item pairs: {len(repeated_pair_summary):,}")
repeated_pair_summary.head(10)

## 5. Timestamp ties and the selected policy

A tied group contains two or more events from one visitor at the exact same millisecond. The file does not tell us their true internal order.

**Selected policy:** exclude every event in a tied group from the modeling copy. This is safer than using raw file row order as a made-up clock. The full tie audit is saved first.

In [ ]:
tie_member = exact_deduped.duplicated(["visitorid", "timestamp"], keep=False)
ties = exact_deduped.loc[tie_member].copy()
ties["source_row_number"] = ties.index + 2  # Original raw-CSV row number; the retained index is preserved after exact-deduplication.
ties = ties.sort_values(["visitorid", "timestamp", "source_row_number"], kind="stable")
ties["tie_group_id"] = ties.groupby(["visitorid", "timestamp"], sort=False).ngroup() + 1
ties["events_in_tie_group"] = ties.groupby("tie_group_id")["tie_group_id"].transform("size")
ties = ties[["tie_group_id", "events_in_tie_group", "source_row_number", *EVENT_COLUMNS]]

tie_group_summary = (
    ties[["tie_group_id", "visitorid", "timestamp", "events_in_tie_group"]]
    .drop_duplicates()
    .sort_values(["events_in_tie_group", "visitorid", "timestamp"], ascending=[False, True, True])
)
ties.to_csv(REPORT_DIR / "04_all_timestamp_ties.csv", index=False)
tie_group_summary.to_csv(REPORT_DIR / "04_timestamp_tie_summary.csv", index=False)

print(f"Tied events: {len(ties):,}")
print(f"Tie groups: {ties['tie_group_id'].nunique():,}")
tie_group_summary.head(10)

In [ ]:
# Final output of this notebook: clean and safe to order chronologically.
modeling_events = exact_deduped.loc[~tie_member].copy()
modeling_events_path = INTERIM_DIR / "events_clean_for_next_steps.csv"
modeling_events.to_csv(modeling_events_path, index=False)

assert modeling_events.duplicated(EVENT_COLUMNS, keep=False).sum() == 0
assert modeling_events.duplicated(["visitorid", "timestamp"], keep=False).sum() == 0

final_check = pd.DataFrame([
    {"check": "raw row count", "value": len(raw)},
    {"check": "exact copies removed", "value": removed_exact_copies},
    {"check": "tied events excluded from modeling copy", "value": int(tie_member.sum())},
    {"check": "final modeling row count", "value": len(modeling_events)},
    {"check": "exact duplicate members remaining", "value": int(modeling_events.duplicated(EVENT_COLUMNS, keep=False).sum())},
    {"check": "timestamp-tie members remaining", "value": int(modeling_events.duplicated(["visitorid", "timestamp"], keep=False).sum())},
])
final_check.to_csv(REPORT_DIR / "05_final_verification.csv", index=False)
final_check

## Manual Excel verification

Open the CSV files in `reports/<RUN_TAG>/`.

- `02_exact_duplicate_audit.csv`: sort `duplicate_group_id`. The first row says `keep_first_source_row`; later identical rows say `remove_exact_duplicate`.
- `03_repeated_visitor_item_summary.csv`: same visitor plus same item at different times is a repeated interaction, not an exact copy.
- `04_all_timestamp_ties.csv`: sort `tie_group_id`; every group has the same visitor and timestamp.
- `05_final_verification.csv`: the last two values must both be zero.

## What comes next

Next notebook: choose the final event policy, apply the iterative user/item ≥5 interaction filter, then build chronologically ordered sequences.